In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
import re
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

from sklearn.preprocessing import OneHotEncoder

data = pd.read_excel("../data/job_classification.ods", engine = "odf", dtype = str)

data.head()

,title,location,description,function,industry,career_level
0,Technical Professional Lead - Process,"Houston, TX","Responsible for the study, design, and specifi...",production_manufacturing,Machinery and Industrial Facilities Engineering,senior_specialist_or_project_manager
1,Cnslt - Systems Eng- Midrange 1,"Seattle, WA","Participates in design, development and implem...",information_technology_telecommunications,Financial Services,senior_specialist_or_project_manager
2,SharePoint Developers and Solution Architects,"Dallas, TX",We are currently in need of Developers who can...,consulting,IT Consulting,senior_specialist_or_project_manager
3,Business Information Services - Strategic Acco...,North Carolina,Experian is seeking an experienced Account Exe...,sales,"Security, Risk, Restructuring Consulting",senior_specialist_or_project_manager
4,Strategic Development Director (procurement),"Austin, TX",Â Want to join a world-class global procuremen...,procurement_materials_logistics,Information Technology,bereichsleiter


In [2]:
data = data.dropna(axis = 0)
data.shape


(8073, 6)

In [3]:
target = "career_level"

x = data.drop(target, axis = 1)
y = data[target]
print (x.shape, y.shape)

(8073, 5) (8073,)


In [4]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42, stratify = y)

print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)

x_train shape: (6458, 5)
y_train shape: (6458,)


In [5]:
def filler_location(location):
    result = re.findall("\\,\\s[A-Z]{2}$", location)
    if len(result) > 0:
        return result[0][2:]
    else:
        return location


data["location"] = data["location"].apply(filler_location)

In [6]:
transformers = ColumnTransformer(transformers = [
    ("title", TfidfVectorizer(stop_words="english"), "title"),
    ("location", OneHotEncoder(handle_unknown="ignore"), ["location"]),
    ("description", TfidfVectorizer(stop_words="english", ngram_range=(1,2)), "description"),
    ("function", OneHotEncoder(handle_unknown="ignore"), ["function"]),
    ("industry", TfidfVectorizer(stop_words="english"), "industry")
])



In [7]:
model = Pipeline(steps = [
    ("transformers", transformers),
    ("classifier", LogisticRegression(max_iter = 1000, class_weight="balanced")),
])

In [ ]:
model.fit(x_train, y_train)

In [ ]:
y_predict = model.predict(x_test)

In [ ]:
print(classification_report(y_test, y_predict))